# Zalopay Complaint Analytics Agent

Run the full complaint analytics pipeline in Google Colab.

**Before running:** Add your secrets in *Colab → Secrets (🔑)* with these keys:
- `LLM_PROVIDER` — `google`, `anthropic`, or `openai`
- `LLM_API_KEY` — your API key
- `JIRA_URL`, `JIRA_EMAIL`, `JIRA_API_TOKEN` *(optional — needed for dry_run=False)*
- `FB_PAGE_ID`, `FB_ACCESS_TOKEN` *(optional — needed for dry_run=False)*
- `THREADS_ACCESS_TOKEN` *(optional — needed for dry_run=False)*

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/aq2208/ai-chay-bang-com-agent"
REPO_DIR = "ai-chay-bang-com-agent"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
!pip install -r requirements.txt -q

## 2. Load credentials from Colab Secrets

Secrets are set once in *Runtime → Secrets* and persist across sessions.
Required: `LLM_PROVIDER`, `LLM_API_KEY`. All others optional (only needed for `dry_run=False`).

In [ ]:
from google.colab import userdata

_SECRETS = [
    ("LLM_PROVIDER",         True),
    ("LLM_API_KEY",          True),
    ("JIRA_URL",             False),
    ("JIRA_EMAIL",           False),
    ("JIRA_API_TOKEN",       False),
    ("FB_PAGE_ID",           False),
    ("FB_ACCESS_TOKEN",      False),
    ("THREADS_ACCESS_TOKEN", False),
]

for key, required in _SECRETS:
    try:
        os.environ[key] = userdata.get(key)
        print(f"  ✅ {key}")
    except Exception:
        if required:
            print(f"  ❌ {key} — REQUIRED but not found in Secrets!")
        else:
            print(f"  ⏭  {key} — not set (only needed for dry_run=False)")

## 3. Build the Knowledge Base index

Run once per session (ChromaDB is stored in-memory in Colab).

In [ ]:
!python knowledge_base/index.py

## 4. Run the pipeline

- `dry_run=True` → uses built-in mock data (no credentials needed, safe to test)
- `dry_run=False` → calls real APIs (requires credentials from Step 2)

In [ ]:
# ── Jira pipeline ─────────────────────────────────────────────────────────
from jobs.jira_job import run as run_jira

result = run_jira(dry_run=True)   # change to False when Jira credentials are ready
print(result)

In [ ]:
# ── Social Media pipeline ──────────────────────────────────────────────────
from jobs.social_job import run as run_social

result = run_social(dry_run=True)   # change to False when FB/Threads credentials are ready
print(result)

## 5. View the generated report

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

reports = sorted(Path("output").glob("*.md"), key=lambda p: p.stat().st_mtime, reverse=True)
if reports:
    display(Markdown(reports[0].read_text()))
else:
    print("No reports found. Run a pipeline job above first.")

## 6. Run tests

Run individual phase test files to verify each component works.

In [ ]:
# Run one at a time — some phases make LLM API calls and consume quota

# !python test_phase2.py   # preprocessor, sentiment, extractor, classifier
# !python test_phase3.py   # knowledge base RAG
# !python test_phase4.py   # image analyzer, grouper
# !python test_phase5.py   # report generator, guardrails
# !python test_phase6.py   # full Jira + Social pipeline (dry_run)
# !python test_phase7.py   # FastAPI endpoints (mocked)

!python test_phase2.py

## 7. Edit code

Open any file from the Colab file browser (left panel → folder icon) and edit directly.
After saving, re-run the relevant cell above — Python modules are reloaded on each `!python` call.

If you edited a module and are importing it in a cell (not via `!python`), restart the runtime
first (*Runtime → Restart runtime*) then re-run from Cell 1.